<a href="https://colab.research.google.com/github/jessicamile94-star/crisis-mmd-multimodal-emergency-detection/blob/main/CrisisMMD_Multimodal_Jessica_Ballesteros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Descarga de Dependencias necesarias

In [7]:
import os
import sys
import json
import time
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Dispositivo: cuda
GPU: Tesla T4


Creacion de carpetas del proyecto

In [8]:
from pathlib import Path

ROOT = Path("/content/crisis-mmd-project")
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
CACHE_DIR = ARTIFACTS_DIR / "cache"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

for folder in [DATA_DIR, ARTIFACTS_DIR, CACHE_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Carpeta principal:", ROOT)
print("Datos:", DATA_DIR)
print("Artefactos:", ARTIFACTS_DIR)

Carpeta principal: /content/crisis-mmd-project
Datos: /content/crisis-mmd-project/data
Artefactos: /content/crisis-mmd-project/artifacts


carga de datos de texto

In [9]:
from huggingface_hub import snapshot_download

HF_REPO = "QCRI/CrisisMMD"
HF_CONFIG = "informative"

snapshot_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    local_dir=str(DATA_DIR),
    allow_patterns=["informative/*.json"],
)

print("Archivos descargados:")
for archivo in sorted((DATA_DIR / HF_CONFIG).glob("*.json")):
    print(archivo.name)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Archivos descargados:
dev.json
test.json
train.json


estratificacion de datos

In [10]:
frames = []

for split_name in ["train", "dev", "test"]:
    file_path = DATA_DIR / "informative" / f"{split_name}.json"
    temp = pd.read_json(file_path)
    temp["official_split"] = split_name
    frames.append(temp)

df_full = pd.concat(frames, ignore_index=True)
sample_parts = []

for (split_name, label), group in df_full.groupby(
    ["official_split", "label"], observed=True
):
    n_sample = max(1, int(len(group) * 0.10))
    sample_parts.append(
        group.sample(n=n_sample, random_state=SEED)
    )

df = pd.concat(sample_parts, ignore_index=True)

print("Dataset completo:", df_full.shape)
print("Muestra estratificada:", df.shape)
print("\nDistribución por partición y clase:")
display(pd.crosstab(df["official_split"], df["label"]))

Dataset completo: (18082, 7)
Muestra estratificada: (1806, 7)

Distribución por partición y clase:


label,informative,not_informative
official_split,,
dev,140,83
test,137,86
train,834,526
